## Load and preprocess Data

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

raw_data_root = os.getcwd()
dataset_folder = os.path.join(raw_data_root, "Top_Bottom_Marked_Data")
turning_points_labeled_file_path = os.path.join(dataset_folder, "AAPL_turning_points_marked.csv")

df = pd.read_csv(turning_points_labeled_file_path)

# Drop the Datetime column (since models don't process dates directly)
df = df.drop(columns=['Datetime'])

print(df['Target_Value'].value_counts())

Target_Value
0    2927
2     232
1     226
Name: count, dtype: int64


In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))
df['Close_Price'] = scaler.fit_transform(df[['Close_Price']])

## Split the Data

In [3]:
X = df[['Close_Price']]  # Features
y = df['Target_Value']  # Labels

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 2708
Testing Samples: 677


## Handle Class Imbalance : ADASYN Oversampling

In [4]:
from imblearn.over_sampling import ADASYN

adasyn = ADASYN()
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

import collections
print("New Training Distribution:", collections.Counter(y_train_resampled))

New Training Distribution: Counter({1: 2353, 0: 2341, 2: 2331})


## Train Model

In [5]:
import xgboost as xgb
from sklearn.metrics import classification_report

model = xgb.XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")
model.fit(X_train_resampled, y_train_resampled)

y_pred = model.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred, target_names=["Normal", "Bottom", "Top"]))

              precision    recall  f1-score   support

      Normal       0.91      0.46      0.61       586
      Bottom       0.12      0.47      0.19        45
         Top       0.11      0.50      0.18        46

    accuracy                           0.46       677
   macro avg       0.38      0.47      0.33       677
weighted avg       0.80      0.46      0.55       677



## Save and Load the Model

In [6]:
import joblib
joblib.dump(model, "turning_point_classifier.pkl")

['turning_point_classifier.pkl']

In [7]:
model = joblib.load("turning_point_classifier.pkl")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Normal", "Bottom", "Top"]))


              precision    recall  f1-score   support

      Normal       0.91      0.46      0.61       586
      Bottom       0.12      0.47      0.19        45
         Top       0.11      0.50      0.18        46

    accuracy                           0.46       677
   macro avg       0.38      0.47      0.33       677
weighted avg       0.80      0.46      0.55       677

